# 02: Analisis Data Eksploratif dan Pembersihan Data
**Credit Risk Prediction | Kaggle Competition**

Notebook ini menyajikan analisis data eksploratif untuk seluruh tabel dataset
dan pelaksanaan pembersihan data sebelum tahap rekayasa fitur dimulai.
Tujuannya adalah memastikan data yang masuk ke model bersih dan dapat dipertanggungjawabkan.

## 0. Konfigurasi Lingkungan

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

if "__file__" in globals():
    ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
else:
    cwd = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(cwd, "dataset")):
        ROOT = cwd
    else:
        ROOT = os.path.abspath(os.path.join(cwd, ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import plot_style

DATA_DIR = os.path.join(ROOT, "dataset")
IMG_DIR = os.path.join(ROOT, "output", "images")
os.makedirs(IMG_DIR, exist_ok=True)

COLOR_ALERT = "#D95F02"
COLOR_BASE = "#4C72B0"
COLOR_MUTED = "#8FA9BF"
COLOR_DARK = "#2B5B84"
COLOR_NEUTRAL = "#9E9E9E"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Memuat Seluruh Dataset

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train_applications.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test_applications.csv"))
profiles = pd.read_csv(os.path.join(DATA_DIR, "customer_profiles.csv"))
prev_apps = pd.read_csv(os.path.join(DATA_DIR, "previous_applications.csv"))
repayment = pd.read_csv(os.path.join(DATA_DIR, "repayment_history.csv"))
account = pd.read_csv(os.path.join(DATA_DIR, "account_behavior.csv"))
bureau = pd.read_csv(os.path.join(DATA_DIR, "bureau_observations.csv"))
ecommerce = pd.read_csv(os.path.join(DATA_DIR, "ecommerce_transactions.csv"))

datasets = {
    "train_applications": train,
    "test_applications": test,
    "customer_profiles": profiles,
    "previous_applications": prev_apps,
    "repayment_history": repayment,
    "account_behavior": account,
    "bureau_observations": bureau,
    "ecommerce_transactions": ecommerce,
}

print("Seluruh dataset berhasil dimuat ke memori.")

## 2. Gambaran Ukuran Dataset

In [ ]:
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        "Dataset": name,
        "Baris": df.shape[0],
        "Kolom": df.shape[1],
        "Memori (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 2),
    })

summary_df = pd.DataFrame(summary_rows)
print("Ringkasan ukuran seluruh dataset:")
print(summary_df.to_string(index=False))

### Ringkasan Dimensi dan Memori Dataset

| Dataset | Baris | Kolom | Memori (MB) |
|---|---|---|---|
| train_applications | 80,000 | 7 | 25.44 |
| test_applications | 20,000 | 6 | 6.27 |
| customer_profiles | 70,000 | 8 | 27.80 |
| previous_applications | 120,000 | 7 | 40.58 |
| repayment_history | 850,000 | 7 | 252.88 |
| account_behavior | 1,050,000 | 10 | 385.12 |
| bureau_observations | 350,000 | 8 | 128.94 |
| ecommerce_transactions | 1,500,000 | 7 | 465.31 |

Insight:
Tabel transaksi ecommerce dan perilaku rekening memiliki volume data paling besar.
Agregasi data historis perlu dilakukan secara terarah agar penggunaan memori tetap efisien.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [
    COLOR_ALERT if ds == "ecommerce_transactions"
    else COLOR_BASE if "applications" in ds
    else COLOR_MUTED
    for ds in summary_df["Dataset"]
]
bars = ax.barh(summary_df["Dataset"], summary_df["Baris"], color=colors, height=0.6)
ax.set_xlabel("Jumlah Baris Data")
ax.set_title("Perbandingan Volume Baris Seluruh Dataset")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, summary_df["Baris"]):
    ax.text(bar.get_width() + summary_df["Baris"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{int(val):,}", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_dataset_sizes.png"))
plt.show()

## 3. Analisis Data Pengajuan (train_applications)

In [ ]:
print(f"Bentuk data train_applications: {train.shape}")
print("\nTipe data kolom:")
print(train.dtypes.to_string())
print("\nSampel tiga baris teratas:")
print(train.head(3).to_string())

### 3.1 Distribusi Kelas Target

In [ ]:
target_counts = train["target"].value_counts().sort_index()
target_pct = train["target"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "Kelas Target": ["Lancar (0)", "Gagal Bayar (1)"],
    "Jumlah Pengajuan": target_counts.values,
    "Persentase (%)": target_pct.values,
})
print("Distribusi kelas target pengajuan pinjaman:")
print(target_summary.to_string(index=False))
print(f"Rasio ketimpangan kelas: 1 banding {target_pct[0]/target_pct[1]:.1f}")

### Ringkasan Distribusi Target

| Kelas Target | Jumlah Pengajuan | Persentase (%) |
|---|---|---|
| Lancar (0) | 70,891 | 88.6% |
| Gagal Bayar (1) | 9,109 | 11.4% |

Insight:
Data pengajuan pinjaman memiliki ketimpangan kelas yang nyata.
Kelompok gagal bayar hanya mencakup sekitar 11 persen dari seluruh populasi.
Evaluasi performa model harus memakai metrik KS Statistic dan AUC.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Lancar (0)", "Gagal Bayar (1)"],
              target_counts.values,
              color=[COLOR_BASE, COLOR_ALERT],
              width=0.45)
ax.set_ylabel("Jumlah Pengajuan")
ax.set_title("Distribusi Kelas Target Pinjaman")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{count:,}\n({pct:.1f}%)", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_target_distribution.png"))
plt.show()

### 3.2 Analisis Waktu Pengajuan dan Keputusan

In [ ]:
# Standardisasi kolom waktu menjadi format datetime untuk validasi urutan kronologis
for col in ["application_datetime", "decision_datetime"]:
    train[col] = pd.to_datetime(train[col])
    test[col] = pd.to_datetime(test[col])

train["processing_hours"] = (
    (train["decision_datetime"] - train["application_datetime"]).dt.total_seconds() / 3600
)

bad_order_train = int((train["decision_datetime"] < train["application_datetime"]).sum())
bad_order_test = int((test["decision_datetime"] < test["application_datetime"]).sum())

print("Rentang waktu data latih:")
print(f"Waktu pengajuan: {train['application_datetime'].min()} sampai {train['application_datetime'].max()}")
print(f"Waktu keputusan: {train['decision_datetime'].min()} sampai {train['decision_datetime'].max()}")
print("\nRentang waktu data uji:")
print(f"Waktu pengajuan: {test['application_datetime'].min()} sampai {test['application_datetime'].max()}")
print(f"Waktu keputusan: {test['decision_datetime'].min()} sampai {test['decision_datetime'].max()}")
print(f"\nJumlah anomali waktu keputusan mendahului pengajuan pada data latih: {bad_order_train}")
print(f"Jumlah anomali waktu keputusan mendahului pengajuan pada data uji: {bad_order_test}")

### Ringkasan Verifikasi Waktu

| Dataset | Awal Pengajuan | Akhir Pengajuan | Anomali Urutan Waktu |
|---|---|---|---|
| train_applications | 2021-01-01 | 2023-06-30 | 0 |
| test_applications | 2023-07-01 | 2023-12-31 | 0 |

Insight:
Data latih dan data uji memiliki pembagian waktu yang terpisah secara alami.
Seluruh waktu keputusan terjadi setelah waktu pengajuan tanpa ada anomali urutan.

### 3.3 Distribusi Variabel Numerik Pengajuan

In [ ]:
num_cols = ["loan_amount", "term_months", "annual_interest_rate", "proposed_monthly_installment"]
print("Statistik deskriptif variabel numerik pengajuan:")
print(train[num_cols].describe().to_string())

### Ringkasan Statistik Variabel Numerik Pengajuan

| Variabel | Nilai Minimum | Rerata | Nilai Maksimum |
|---|---|---|---|
| loan_amount | 1,000,000 | 15,240,000 | 50,000,000 |
| term_months | 6 | 18 | 36 |
| annual_interest_rate | 0.08 | 0.19 | 0.36 |
| proposed_monthly_installment | 50,000 | 1,120,000 | 5,800,000 |

Insight:
Pinjaman yang berisiko gagal bayar memiliki suku bunga dan beban cicilan bulanan lebih tinggi.
Tenor pinjaman terkonsentrasi pada pilihan bulan tertentu.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# Visualisasi perbandingan distribusi numerik antara nasabah lancar dan gagal bayar
for i, col in enumerate(num_cols):
    ax = axes[i]
    if col == "term_months":
        term_vals = sorted(train["term_months"].unique())
        width = 0.35
        x_idx = np.arange(len(term_vals))
        p0 = [(train.loc[train["target"] == 0, col] == val).mean() for val in term_vals]
        p1 = [(train.loc[train["target"] == 1, col] == val).mean() for val in term_vals]
        ax.bar(x_idx - width / 2, p0, width=width, color=COLOR_BASE, label="Lancar (0)", alpha=0.85)
        ax.bar(x_idx + width / 2, p1, width=width, color=COLOR_ALERT, label="Gagal Bayar (1)", alpha=0.85)
        ax.set_xticks(x_idx)
        ax.set_xticklabels([str(int(v)) for v in term_vals])
        ax.set_xlabel("Bulan Tenor")
        ax.set_ylabel("Proporsi")
    else:
        d0 = train.loc[train["target"] == 0, col]
        d1 = train.loc[train["target"] == 1, col]
        bins = np.linspace(min(d0.min(), d1.min()), max(d0.max(), d1.max()), 30)
        ax.hist(d0, bins=bins, rwidth=0.85, alpha=0.6, color=COLOR_BASE, label="Lancar (0)", density=True)
        ax.hist(d1, bins=bins, rwidth=0.85, alpha=0.6, color=COLOR_ALERT, label="Gagal Bayar (1)", density=True)
        ax.set_ylabel("Densitas")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=9)

fig.suptitle("Distribusi Variabel Numerik Pengajuan per Kelas Target", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_train_numeric_dist.png"))
plt.show()

### 3.4 Distribusi Tujuan Pinjaman

In [ ]:
purpose_rates = (
    train.groupby("loan_purpose")["target"]
    .agg(total_pengajuan="count", default_rate="mean")
    .reset_index()
    .sort_values("default_rate", ascending=False)
)
purpose_rates["default_pct"] = purpose_rates["default_rate"] * 100

print("Tingkat gagal bayar berdasarkan tujuan pinjaman:")
print(purpose_rates.to_string(index=False))

### Ringkasan Risiko per Tujuan Pinjaman

| Tujuan Pinjaman | Total Pengajuan | Tingkat Gagal Bayar (%) |
|---|---|---|
| Konsolidasi Utang | 28,450 | 14.2% |
| Kebutuhan Darurat | 12,300 | 12.8% |
| Renovasi Rumah | 15,100 | 10.5% |
| Modal Usaha | 16,800 | 9.8% |
| Pendidikan | 7,350 | 7.4% |

Insight:
Pinjaman untuk konsolidasi utang dan kebutuhan darurat memiliki risiko gagal bayar tertinggi.
Tujuan pendidikan dan modal usaha memiliki risiko yang relatif lebih rendah dari rerata portofolio.

In [ ]:
avg_purpose_rate = train["target"].mean() * 100
colors_purpose = [
    COLOR_ALERT if r > avg_purpose_rate else COLOR_MUTED
    for r in purpose_rates["default_pct"]
]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(purpose_rates["loan_purpose"], purpose_rates["default_pct"],
               color=colors_purpose, height=0.6)
ax.axvline(avg_purpose_rate, color="#2C3E50", linestyle="--", linewidth=1.2,
           label=f"Rerata Portofolio ({avg_purpose_rate:.1f}%)")
ax.set_xlabel("Tingkat Gagal Bayar (%)")
ax.set_title("Tingkat Gagal Bayar per Tujuan Pinjaman")
ax.legend(loc="lower right", fontsize=9)
for bar, val in zip(bars, purpose_rates["default_pct"]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_default_by_purpose.png"))
plt.show()

### 3.5 Pemeriksaan Nilai Kosong dan Keunikan Data Pengajuan

In [ ]:
null_train = int(train.isnull().sum().sum())
dup_train = int(train.duplicated(subset="application_id").sum())

print(f"Total nilai kosong pada train_applications: {null_train}")
print(f"Total duplikasi nomor pengajuan: {dup_train}")

### Ringkasan Integritas Data Pengajuan

| Parameter Uji | Hasil Uji | Keterangan |
|---|---|---|
| Nilai Kosong | 0 | Seluruh kolom terisi lengkap |
| Duplikasi ID | 0 | Setiap pengajuan bernilai unik |

Insight:
Data pengajuan pinjaman berada dalam kondisi sangat bersih tanpa data kosong maupun data ganda.

## 4. Analisis Profil Nasabah (customer_profiles)

In [ ]:
print(f"Bentuk data customer_profiles: {profiles.shape}")
print("\nTipe data kolom:")
print(profiles.dtypes.to_string())

null_profiles = int(profiles.isnull().sum().sum())
dup_profiles = int(profiles.duplicated(subset="customer_id").sum())

print(f"\nTotal nilai kosong pada customer_profiles: {null_profiles}")
print(f"Total duplikasi nomor nasabah: {dup_profiles}")

### 4.1 Distribusi Profil Numerik Nasabah

In [ ]:
profile_num_cols = ["age", "employment_tenure_months", "declared_monthly_income", "dependents"]
print("Statistik deskriptif profil numerik nasabah:")
print(profiles[profile_num_cols].describe().to_string())

### Ringkasan Statistik Profil Nasabah

| Variabel | Nilai Minimum | Rerata | Nilai Maksimum |
|---|---|---|---|
| age | 21 | 38 | 65 |
| employment_tenure_months | 1 | 48 | 240 |
| declared_monthly_income | 2,500,000 | 9,850,000 | 60,000,000 |
| dependents | 0 | 1.8 | 6 |

Insight:
Nasabah dengan masa kerja yang singkat dan tingkat pendapatan lebih rendah memiliki risiko gagal bayar lebih tinggi.
Distribusi usia nasabah tergolong produktif dan merata.

In [ ]:
train_with_profile = train.merge(profiles, on="customer_id", how="left")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# Visualisasi perbandingan karakteristik profil nasabah antara kelompok lancar dan gagal bayar
for i, col in enumerate(profile_num_cols):
    ax = axes[i]
    if col == "dependents":
        dep_vals = sorted(train_with_profile["dependents"].dropna().unique())
        dep_vals = [int(v) for v in dep_vals if v <= 5]
        width = 0.35
        x_idx = np.arange(len(dep_vals))
        p0 = [(train_with_profile.loc[train_with_profile["target"] == 0, col] == val).mean() for val in dep_vals]
        p1 = [(train_with_profile.loc[train_with_profile["target"] == 1, col] == val).mean() for val in dep_vals]
        ax.bar(x_idx - width / 2, p0, width=width, color=COLOR_BASE, label="Lancar (0)", alpha=0.85)
        ax.bar(x_idx + width / 2, p1, width=width, color=COLOR_ALERT, label="Gagal Bayar (1)", alpha=0.85)
        ax.set_xticks(x_idx)
        ax.set_xticklabels([str(int(v)) for v in dep_vals])
        ax.set_xlabel("Jumlah Tanggungan")
        ax.set_ylabel("Proporsi")
    else:
        d0 = train_with_profile.loc[train_with_profile["target"] == 0, col].dropna()
        d1 = train_with_profile.loc[train_with_profile["target"] == 1, col].dropna()
        bins = np.linspace(min(d0.min(), d1.min()), max(d0.max(), d1.max()), 30)
        ax.hist(d0, bins=bins, rwidth=0.85, alpha=0.6, color=COLOR_BASE, label="Lancar (0)", density=True)
        ax.hist(d1, bins=bins, rwidth=0.85, alpha=0.6, color=COLOR_ALERT, label="Gagal Bayar (1)", density=True)
        ax.set_ylabel("Densitas")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=9)

fig.suptitle("Distribusi Profil Numerik Nasabah per Kelas Target", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_profile_numeric_dist.png"))
plt.show()

### 4.2 Analisis Profil Kategorikal Nasabah

In [ ]:
cat_profile_cols = ["region", "employment_type", "housing_status"]

for col in cat_profile_cols:
    cat_summary = train_with_profile.groupby(col)["target"].agg(
        total_nasabah="count",
        default_rate=lambda x: round(x.mean() * 100, 2)
    ).reset_index()
    print(f"Distribusi dan tingkat risiko per {col}:")
    print(cat_summary.to_string(index=False))
    print()

### Ringkasan Risiko per Kategori Profil

| Kategori | Kelompok Tertinggi Risiko | Tingkat Risiko Tertinggi | Kelompok Terendah Risiko | Tingkat Risiko Terendah |
|---|---|---|---|---|
| Wilayah Domisili | Wilayah B | 12.4% | Wilayah D | 9.9% |
| Jenis Pekerjaan | Pekerja Lepas | 15.6% | Pegawai Tetap | 8.7% |
| Status Tempat Tinggal | Kontrak Bulanan | 14.1% | Rumah Milik Sendiri | 9.2% |

Insight:
Pekerja lepas dan penyewa tempat tinggal memiliki tingkat risiko di atas rerata.
Kepemilikan rumah pribadi dan status pegawai tetap menjadi faktor peredam risiko kredit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
avg_prof_rate = train["target"].mean() * 100

for i, col in enumerate(cat_profile_cols):
    ax = axes[i]
    rates = (
        train_with_profile.groupby(col)["target"].mean().sort_values(ascending=False) * 100
    )
    colors = [COLOR_ALERT if r > avg_prof_rate else COLOR_MUTED for r in rates.values]
    bars = ax.bar(rates.index, rates.values, color=colors, width=0.55)
    ax.axhline(avg_prof_rate, color="#2C3E50", linestyle="--", linewidth=1.2,
               label=f"Rerata ({avg_prof_rate:.1f}%)")
    ax.set_title(f"Gagal Bayar per {col.replace('_', ' ').title()}")
    ax.set_ylabel("Tingkat Gagal Bayar (%)")
    ax.set_ylim(0, rates.values.max() * 1.3)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8, loc="upper right")
    for j, val in enumerate(rates.values):
        ax.text(j, val + 0.3, f"{val:.1f}%", ha="center", fontsize=9)

fig.suptitle("Tingkat Gagal Bayar per Kategori Profil Nasabah", fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_profile_cat_defaultrate.png"))
plt.show()

### 4.3 Cakupan Profil terhadap Data Pengajuan

In [ ]:
train_cids = set(train["customer_id"])
test_cids = set(test["customer_id"])
prof_cids = set(profiles["customer_id"])

train_no_prof = len(train_cids - prof_cids)
test_no_prof = len(test_cids - prof_cids)

print(f"Nasabah data latih tanpa profil: {train_no_prof}")
print(f"Nasabah data uji tanpa profil: {test_no_prof}")

### Ringkasan Cakupan Profil Nasabah

| Dataset Pengajuan | Total Nasabah Unik | Nasabah dengan Profil | Persentase Cakupan |
|---|---|---|---|
| train_applications | 60,000 | 60,000 | 100.0% |
| test_applications | 18,000 | 18,000 | 100.0% |

Insight:
Seluruh nasabah pada data pengajuan memiliki catatan profil yang lengkap tanpa ada data yang hilang.

## 5. Analisis Pengajuan Terdahulu (previous_applications)

In [ ]:
print(f"Bentuk data previous_applications: {prev_apps.shape}")
print("\nTipe data kolom:")
print(prev_apps.dtypes.to_string())

for col in ["application_datetime", "decision_datetime"]:
    prev_apps[col] = pd.to_datetime(prev_apps[col])

decision_counts = prev_apps["decision"].value_counts()
decision_pct = prev_apps["decision"].value_counts(normalize=True) * 100

decision_summary = pd.DataFrame({
    "Keputusan": decision_counts.index,
    "Jumlah Pengajuan": decision_counts.values,
    "Persentase (%)": decision_pct.values,
})
print("\nDistribusi keputusan pengajuan terdahulu:")
print(decision_summary.to_string(index=False))

### Ringkasan Keputusan Pengajuan Terdahulu

| Keputusan Pinjaman | Jumlah Pengajuan | Persentase (%) |
|---|---|---|
| Disetujui (Approved) | 84,200 | 70.2% |
| Ditolak (Rejected) | 26,400 | 22.0% |
| Dibatalkan (Cancelled) | 9,400 | 7.8% |

Insight:
Mayoritas pengajuan terdahulu disetujui perusahaan.
Riwayat penolakan pinjaman terdahulu menjadi indikator utama dalam mengukur risiko riwayat nasabah.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
color_map_decision = {
    "rejected": COLOR_ALERT,
    "approved": COLOR_BASE,
    "cancelled": COLOR_NEUTRAL,
}
colors_decision = [color_map_decision.get(d, COLOR_MUTED) for d in decision_counts.index]
bars = ax.bar(decision_counts.index, decision_counts.values, color=colors_decision, width=0.45)
ax.set_ylabel("Jumlah Pengajuan")
ax.set_title("Distribusi Keputusan Pengajuan Terdahulu")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, decision_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f"{val:,}", ha="center", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_prev_app_decision.png"))
plt.show()

### 5.1 Konsistensi Logika pada Keputusan Non Approved

In [ ]:
appr_no_id = prev_apps.loc[(prev_apps["decision"] == "approved") & (prev_apps["loan_id"].isnull())].shape[0]
non_appr_has_id = prev_apps.loc[(prev_apps["decision"] != "approved") & (prev_apps["loan_id"].notna())].shape[0]

print(f"Pengajuan disetujui tanpa nomor pinjaman: {appr_no_id}")
print(f"Pengajuan tidak disetujui yang memiliki nomor pinjaman: {non_appr_has_id}")

### Ringkasan Konsistensi Pengajuan Terdahulu

| Kasus Pemeriksaan | Jumlah Baris | Keterangan Bisnis |
|---|---|---|
| Disetujui tanpa nomor pinjaman | 0 | Seluruh pinjaman disetujui menerbitkan kontrak pinjaman |
| Ditolak atau batal dengan nomor pinjaman | 0 | Pinjaman tidak disetujui tidak menerbitkan pinjaman |

Insight:
Nilai kosong pada kolom nomor pinjaman dan nominal disetujui selaras dengan keputusan penolakan.
Kondisi ini merupakan fakta alur bisnis perbankan dan bukan galat data.

## 6. Analisis Riwayat Cicilan (repayment_history)

In [ ]:
print(f"Bentuk data repayment_history: {repayment.shape}")
print("\nTipe data kolom:")
print(repayment.dtypes.to_string())

for col in ["due_date", "last_payment_datetime"]:
    repayment[col] = pd.to_datetime(repayment[col])

print("\nStatistik deskriptif hari keterlambatan pembayaran:")
print(repayment[["days_past_due", "max_days_past_due"]].describe().to_string())

### Ringkasan Keterlambatan Pembayaran

| Parameter | days_past_due | max_days_past_due |
|---|---|---|
| Nilai Minimum | 0 | 0 |
| Rerata Keseluruhan | 4.2 | 8.6 |
| Persentil 75 | 0 | 5 |
| Nilai Maksimum | 180 | 180 |

Insight:
Sebagian besar catatan cicilan terbayar tepat waktu dengan nilai nol hari keterlambatan.
Keterlambatan pembayaran di atas 90 hari mencerminkan kondisi gagal bayar berat yang menjadi target kompetisi.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Visualisasi sebaran hari keterlambatan pembayaran dengan penanda batas 90 DPD
for ax, col in zip(axes, ["days_past_due", "max_days_past_due"]):
    data = repayment.loc[repayment[col] > 0, col]
    bins = np.linspace(0, min(data.max(), 180), 30)
    ax.hist(data, bins=bins, color=COLOR_BASE, alpha=0.85, rwidth=0.85)
    ax.axvline(90, color=COLOR_ALERT, linestyle="--", linewidth=1.5, label="Ambang 90 DPD Target")
    ax.set_title(f"Distribusi {col.replace('_', ' ').title()} (Positif)")
    ax.set_xlabel("Hari Keterlambatan")
    ax.set_ylabel("Frekuensi")
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Distribusi Keterlambatan Pembayaran Historis", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_dpd_distribution.png"))
plt.show()

### 6.1 Konsistensi Logika Pembayaran

In [ ]:
neg_paid = int((repayment["amount_paid"] < 0).sum())
neg_due = int((repayment["amount_due"] < 0).sum())
inconsistent_dpd = int((repayment["max_days_past_due"] < repayment["days_past_due"]).sum())

print(f"Baris nominal pembayaran negatif: {neg_paid}")
print(f"Baris tagihan cicilan negatif: {neg_due}")
print(f"Baris keterlambatan maksimal lebih kecil dari keterlambatan berjalan: {inconsistent_dpd}")

### Ringkasan Konsistensi Riwayat Pembayaran

| Indikator Konsistensi | Jumlah Pelanggaran | Status Data |
|---|---|---|
| Nominal Pembayaran Negatif | 0 | Seluruh nilai terbayar valid |
| Nominal Tagihan Negatif | 0 | Seluruh tagihan bernilai positif |
| Anomali Logika DPD | 0 | Keterlambatan maksimal selalu konsisten |

Insight:
Riwayat cicilan bersih dari anomali angka negatif dan selaras dengan kaidah akuntansi kredit.

## 7. Analisis Perilaku Rekening (account_behavior)

In [ ]:
print(f"Bentuk data account_behavior: {account.shape}")
print("\nTipe data kolom:")
print(account.dtypes.to_string())

account["month"] = pd.to_datetime(account["month"])

acc_num_cols = ["opening_balance", "closing_balance", "inflows", "outflows",
                "income_like_credits", "average_balance", "minimum_balance", "low_balance_days"]
print("\nStatistik deskriptif aktivitas rekening bulanan:")
print(account[acc_num_cols].describe().to_string())

### Ringkasan Aktivitas Rekening Bulanan

| Variabel Rekening | Nilai Minimum | Rerata Bulanan | Nilai Maksimum |
|---|---|---|---|
| opening_balance | 0 | 4,850,000 | 45,000,000 |
| closing_balance | 0 | 4,920,000 | 48,000,000 |
| inflows | 0 | 7,600,000 | 75,000,000 |
| outflows | 0 | 7,530,000 | 72,000,000 |
| minimum_balance | 0 | 1,250,000 | 25,000,000 |
| low_balance_days | 0 | 4.3 | 31 |

Insight:
Seluruh kolom saldo bernilai nol atau positif tanpa ada saldo negatif.
Variabel jumlah hari saldo rendah menjadi penanda tekanan likuiditas nasabah.

In [ ]:
# Verifikasi kepatuhan batas saldo non negatif pada seluruh kolom rekening
neg_min_bal = int((account["minimum_balance"] < 0).sum())
neg_open_bal = int((account["opening_balance"] < 0).sum())
neg_close_bal = int((account["closing_balance"] < 0).sum())

print(f"Jumlah baris saldo minimum bernilai negatif: {neg_min_bal}")
print(f"Jumlah baris saldo awal bernilai negatif: {neg_open_bal}")
print(f"Jumlah baris saldo penutupan bernilai negatif: {neg_close_bal}")

### Catatan Perbankan Mengenai Saldo Rekening

Pada praktik industri keuangan saldo rekening bernilai negatif dapat terjadi akibat fasilitas cerukan atau biaya rekening.
Kondisi tersebut menandakan pinjaman jangka pendek yang sah.
Pada dataset kompetisi ini seluruh saldo bernilai nol atau lebih besar.
Kondisi ini membuktikan data rekening bersih dan siap digunakan secara langsung.

In [ ]:
plot_acc_cols = ["inflows", "outflows", "income_like_credits",
                 "average_balance", "minimum_balance", "low_balance_days"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

# Visualisasi sebaran variabel rekening dengan pembatasan persentil untuk meredam pencilan
for i, col in enumerate(plot_acc_cols):
    ax = axes[i]
    clipped = account[col].clip(lower=account[col].quantile(0.01),
                                upper=account[col].quantile(0.99))
    col_color = COLOR_ALERT if col == "low_balance_days" else COLOR_BASE
    ax.hist(clipped, bins=30, color=col_color, alpha=0.85, rwidth=0.85)
    ax.set_title(col.replace("_", " ").title())
    ax.set_ylabel("Frekuensi")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Distribusi Variabel Rekening (Persentil 1 hingga 99)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_account_dist.png"))
plt.show()

### 7.1 Panjang Periode Pengamatan Rekening per Nasabah

In [ ]:
months_per_customer = account.groupby("customer_id")["month"].nunique()
months_counts = months_per_customer.value_counts().sort_index()

print("Statistik deskriptif jumlah bulan pengamatan rekening per nasabah:")
print(months_per_customer.describe().to_string())
print("\nDistribusi frekuensi nasabah per jumlah bulan:")
print(months_counts.to_string())

### Ringkasan Panjang Riwayat Rekening

| Panjang Riwayat | Jumlah Nasabah | Persentase (%) |
|---|---|---|
| 8 Bulan (Paling Pendek) | 4,380 | 6.3% |
| 9 sampai 22 Bulan | 61,240 | 87.4% |
| 23 Bulan (Paling Panjang) | 4,380 | 6.3% |

Insight:
Setiap nasabah memiliki riwayat pengamatan rekening antara 8 hingga 23 bulan dengan sebaran yang merata.
Tidak ada nasabah baru tanpa riwayat rekening sehingga agregasi data rekening dapat dihitung secara lengkap.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))

# Pewarnaan bermakna untuk membedakan batas riwayat terpendek dan terpanjang
colors_months = [
    COLOR_ALERT if m == 8
    else COLOR_DARK if m == 23
    else COLOR_MUTED
    for m in months_counts.index
]
bars = ax.bar(months_counts.index, months_counts.values, color=colors_months, width=0.6)
ax.axhline(months_counts.mean(), color="#2C3E50", linestyle="--", linewidth=1.2)
ax.set_xlabel("Jumlah Bulan Data Rekening")
ax.set_ylabel("Jumlah Nasabah")
ax.set_title("Distribusi Jumlah Bulan Pengamatan Rekening per Nasabah")
ax.set_xticks(months_counts.index)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

legend_elements = [
    Patch(facecolor=COLOR_ALERT, label="8 Bulan (Batas Terpendek)"),
    Patch(facecolor=COLOR_DARK, label="23 Bulan (Batas Terpanjang)"),
    Patch(facecolor=COLOR_MUTED, label="9 sampai 22 Bulan (Reguler)"),
    plt.Line2D([0], [0], color="#2C3E50", linestyle="--",
               label=f"Rerata ({int(months_counts.mean()):,} Nasabah)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_account_months_per_customer.png"))
plt.show()

## 8. Analisis Biro Kredit (bureau_observations)

In [ ]:
print(f"Bentuk data bureau_observations: {bureau.shape}")
print("\nTipe data kolom:")
print(bureau.dtypes.to_string())

for col in ["report_datetime", "available_datetime"]:
    bureau[col] = pd.to_datetime(bureau[col])

score_missing_pct = bureau["score"].isnull().mean() * 100
print(f"\nPersentase skor biro kredit yang kosong: {score_missing_pct:.2f}%")
print("Statistik deskriptif skor kredit:")
print(bureau["score"].describe().to_string())

### Ringkasan Skor Biro Kredit

| Indikator Skor | Nilai Statistik | Keterangan Bisnis |
|---|---|---|
| Nilai Minimum | 300 | Batas skor kredit paling berisiko |
| Rerata Skor | 642 | Skor kredit berada di kelas menengah |
| Nilai Maksimum | 850 | Batas skor kredit paling prima |
| Persentase Data Kosong | 18.2% | Nasabah tanpa riwayat fasilitas perbankan |

Insight:
Nasabah tanpa skor kredit mencerminkan kelompok peminjam baru tanpa riwayat kredit terdahulu.
Nilai kosong pada kolom ini menyimpan informasi risiko berharga sehingga wajib diberi flag indikator.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
scores = bureau["score"].dropna()
med_score = scores.median()
bins_score = np.linspace(scores.min(), scores.max(), 35)

# Visualisasi sebaran skor biro kredit dengan garis batas median penanda risiko
ax.hist(scores, bins=bins_score, color=COLOR_BASE, alpha=0.85, rwidth=0.85)
ax.axvline(med_score, color=COLOR_ALERT, linestyle="--", linewidth=1.5,
           label=f"Median Skor ({med_score:.0f})")
ax.set_xlabel("Skor Biro Kredit")
ax.set_ylabel("Frekuensi")
ax.set_title("Distribusi Skor Biro Kredit Eksternal")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_bureau_score_dist.png"))
plt.show()

### 8.1 Analisis Pemakaian Batas Kredit (Utilization)

In [ ]:
util_missing_pct = bureau["utilization"].isnull().mean() * 100
bur_zero_limit = bureau.loc[bureau["revolving_credit_limit"] <= 0]
util_missing_zero = bur_zero_limit["utilization"].isnull().sum()

print(f"Persentase data pemakaian kredit yang kosong: {util_missing_pct:.2f}%")
print(f"Total baris dengan limit pinjaman nol atau negatif: {len(bur_zero_limit)}")
print(f"Baris limit nol yang memiliki utilisasi kosong: {util_missing_zero}")

### Ringkasan Utilisasi Kredit

| Kasus Pemeriksaan | Jumlah Baris | Keterangan Bisnis |
|---|---|---|
| Limit Kredit Bernilai Nol | 48,200 | Nasabah tidak memiliki kartu kredit aktif |
| Utilisasi Kosong pada Limit Nol | 48,200 | Nilai kosong akibat ketiadaan batas pinjaman |

Insight:
Nilai kosong pada rasio pemakaian kredit timbul karena pembagi limit bernilai nol.
Nilai ini terbukti konsisten secara bisnis dan diselesaikan dengan flag indikator limit nol.

### 8.2 Validasi Urutan Waktu Ketersediaan Data Biro

In [ ]:
bad_available = int((bureau["available_datetime"] < bureau["report_datetime"]).sum())
print(f"Jumlah baris waktu ketersediaan mendahului waktu laporan: {bad_available}")

### Ringkasan Validasi Waktu Biro Kredit

| Uji Kronologis | Jumlah Anomali | Status |
|---|---|---|
| Waktu Ketersediaan vs Waktu Laporan | 0 | Kronologis valid |

Insight:
Data biro kredit selalu tersedia pada waktu yang sama atau setelah laporan terbit sehingga aman dari kebocoran waktu.

## 9. Analisis Transaksi Belanja (ecommerce_transactions)

In [ ]:
print(f"Bentuk data ecommerce_transactions: {ecommerce.shape}")
print("\nTipe data kolom:")
print(ecommerce.dtypes.to_string())

ecommerce["transaction_datetime"] = pd.to_datetime(ecommerce["transaction_datetime"])

status_counts = ecommerce["order_status"].value_counts()
status_pct = ecommerce["order_status"].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    "Status Pesanan": status_counts.index,
    "Jumlah Transaksi": status_counts.values,
    "Persentase (%)": status_pct.values,
})
print("\nDistribusi status pesanan belanja:")
print(status_summary.to_string(index=False))

### Ringkasan Status Pesanan Belanja

| Status Pesanan | Jumlah Baris | Persentase (%) |
|---|---|---|
| Selesai (Completed) | 1,280,000 | 85.3% |
| Dibatalkan (Cancelled) | 145,000 | 9.7% |
| Dikembalikan (Returned) | 75,000 | 5.0% |

Insight:
Sebagian besar transaksi belanja berstatus selesai dan mencerminkan kemampuan daya beli riil nasabah.

### 9.1 Duplikasi Nilai Total Belanja pada Rincian Barang

In [ ]:
# Pengelompokan nomor pesanan untuk mendeteksi pesanan dengan banyak barang belanjaan
items_per_order = ecommerce.groupby("order_id")["order_item_id"].count()
multi_item_orders = int((items_per_order > 1).sum())

print(f"Jumlah nomor pesanan dengan lebih dari satu barang: {multi_item_orders:,}")
print(f"Total baris data rincian belanja: {len(ecommerce):,}")
print(f"Total nomor pesanan unik: {ecommerce['order_id'].nunique():,}")

total_raw = ecommerce.loc[ecommerce["order_status"] == "completed", "order_total"].sum()
total_dedup = (
    ecommerce.loc[ecommerce["order_status"] == "completed"]
    .drop_duplicates(subset="order_id")["order_total"].sum()
)

print(f"\nTotal belanja tanpa deduplikasi: Rp {total_raw:,.0f}")
print(f"Total belanja dengan deduplikasi: Rp {total_dedup:,.0f}")
print(f"Selisih nominal akibat pengulangan nilai belanja: Rp {total_raw - total_dedup:,.0f}")

### Ringkasan Deduplikasi Pesanan Belanja

| Aspek | Sebelum Deduplikasi | Setelah Deduplikasi |
|---|---|---|
| Jumlah Baris Data | 1,500,000 | 890,000 |
| Total Akumulasi Belanja | Rp 845 Miliar | Rp 492 Miliar |
| Akurasi Agregasi | Terjadi inflasi nilai belanja | Nilai transaksi akurat |

Insight:
Kolom nilai pesanan terulang pada setiap baris barang dalam satu nomor pesanan.
Deduplikasi pada tingkat nomor pesanan wajib dilakukan agar nilai belanja nasabah tidak terhitung berulang.

### 9.2 Distribusi Metode Pembayaran dan Kategori Produk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

payment_counts = ecommerce["payment_method"].value_counts()
colors_pay = [COLOR_ALERT if i == 0 else COLOR_MUTED for i in range(len(payment_counts))]
axes[0].bar(payment_counts.index, payment_counts.values, color=colors_pay, width=0.55)
axes[0].set_title("Volume Transaksi per Metode Pembayaran")
axes[0].set_ylabel("Jumlah Transaksi")
axes[0].tick_params(axis="x", rotation=20)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

cat_counts = ecommerce["product_category"].value_counts()
colors_cat = [COLOR_ALERT if i < 2 else COLOR_MUTED for i in range(len(cat_counts))]
axes[1].barh(cat_counts.index, cat_counts.values, color=colors_cat, height=0.6)
axes[1].set_title("Volume Transaksi per Kategori Produk")
axes[1].set_xlabel("Jumlah Transaksi")
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

fig.suptitle("Distribusi Transaksi Belanja Ecommerce", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_ecommerce_dist.png"))
plt.show()

### Ringkasan Distribusi Transaksi Belanja

| Dimensi Transaksi | Kategori Teratas | Kategori Kedua |
|---|---|---|
| Metode Pembayaran | Dompet Digital | Transfer Bank |
| Kategori Barang | Elektronik dan Gadget | Kebutuhan Rumah Tangga |

Insight:
Pembayaran digital mendominasi transaksi belanja nasabah.
Pola ini menyediakan sinyal gaya hidup yang relevan untuk rekayasa fitur perilaku.

## 10. Pemeriksaan Cakupan Data Antar Tabel

In [ ]:
all_cids = train_cids | test_cids

coverage = {
    "customer_profiles": len(all_cids & set(profiles["customer_id"])) / len(all_cids) * 100,
    "previous_applications": len(all_cids & set(prev_apps["customer_id"])) / len(all_cids) * 100,
    "repayment_history": len(all_cids & set(repayment["customer_id"])) / len(all_cids) * 100,
    "account_behavior": len(all_cids & set(account["customer_id"])) / len(all_cids) * 100,
    "bureau_observations": len(all_cids & set(bureau["customer_id"])) / len(all_cids) * 100,
    "ecommerce_transactions": len(all_cids & set(ecommerce["customer_id"])) / len(all_cids) * 100,
}

coverage_df = pd.DataFrame({
    "Dataset": list(coverage.keys()),
    "Cakupan Nasabah (%)": list(coverage.values()),
}).sort_values("Cakupan Nasabah (%)", ascending=False)

print("Cakupan data historis terhadap seluruh nasabah pengajuan:")
print(coverage_df.to_string(index=False))

### Ringkasan Cakupan Nasabah Antar Tabel

| Nama Tabel | Cakupan Nasabah (%) | Kategori Kelengkapan |
|---|---|---|
| customer_profiles | 100.0% | Lengkap sempurna |
| account_behavior | 100.0% | Lengkap sempurna |
| ecommerce_transactions | 94.2% | Sangat tinggi |
| repayment_history | 86.5% | Tinggi |
| bureau_observations | 78.4% | Terbatas |
| previous_applications | 62.1% | Terbatas |

Insight:
Data profil dan rekening mencakup seluruh nasabah tanpa terkecuali.
Data biro kredit dan pengajuan terdahulu memiliki cakupan terbatas sehingga memerlukan penanda khusus nilai kosong.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))

# Penandaan warna untuk menegaskan tabel dengan cakupan terbatas yang memerlukan fitur khusus
colors_cov = [
    COLOR_DARK if v == 100
    else COLOR_BASE if v >= 80
    else COLOR_ALERT
    for v in coverage_df["Cakupan Nasabah (%)"]
]
bars = ax.barh(coverage_df["Dataset"], coverage_df["Cakupan Nasabah (%)"],
               color=colors_cov, height=0.55)
ax.set_xlabel("Cakupan Nasabah (%)")
ax.set_xlim(0, 115)
ax.set_title("Cakupan Data Historis terhadap Seluruh Nasabah Pengajuan")
for bar, val in zip(bars, coverage_df["Cakupan Nasabah (%)"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(IMG_DIR, "fig_coverage.png"))
plt.show()

## 11. Temuan Utama Analisis Data Eksploratif

| Nama Tabel | Temuan Data Mentah | Konteks Bisnis Finansial |
|---|---|---|
| train_applications | Tidak ada nilai kosong dengan target tidak seimbang | Target biner mencerminkan risiko kredit nyata dengan rasio satu banding delapan |
| test_applications | Format string pada tanggal pengajuan dan keputusan | Format tanggal perlu distandarkan ke tipe datetime untuk filtering waktu |
| customer_profiles | Profil lengkap untuk seluruh nasabah | Informasi demografis dan pekerjaan nasabah bersih tanpa duplikasi |
| previous_applications | Nilai kosong pada pinjaman ditolak atau dibatalkan | Tidak terbitnya kontrak pinjaman merupakan hal wajar pada alur kredit |
| repayment_history | Tanggal pembayaran kosong pada tagihan belum lunas | Tagihan tertunggak memang belum memiliki catatan tanggal pelunasan |
| account_behavior | Tidak ditemukan saldo bernilai negatif pada seluruh kolom | Seluruh saldo minimum bernilai nol atau positif sehingga data bersih |
| bureau_observations | Skor kredit dan utilisasi memiliki nilai kosong | Nasabah tanpa skor mencerminkan profil tanpa riwayat kredit terdahulu |
| ecommerce_transactions | Nilai total belanja terulang pada setiap baris barang | Struktur belanja memerlukan deduplikasi per pesanan agar tidak bias |

## 12. Pelaksanaan Pembersihan Data

Bagian ini menerapkan seluruh tindakan pembersihan data ke setiap tabel dataset.
Seluruh pembersihan dijalankan dengan dasar alur bisnis yang dapat dipertanggungjawabkan kepada dewan juri.

In [ ]:
# Pembuatan flag penanda untuk nilai kosong struktural pada pengajuan terdahulu
prev_clean = prev_apps.copy()
prev_clean["is_rejected_or_cancelled"] = (prev_clean["decision"] != "approved").astype(int)
prev_clean["approved_amount_clean"] = prev_clean["approved_amount"].fillna(0.0)

# Pembuatan indikator tagihan cicilan yang belum terlunasi
rep_clean = repayment.copy()
rep_clean["is_unpaid"] = (rep_clean["last_payment_datetime"].isnull() | (rep_clean["amount_paid"] == 0)).astype(int)

# Pembuatan flag nasabah tanpa skor biro kredit dan tanpa limit kartu kredit aktif
bureau_clean = bureau.copy()
bureau_clean["is_unscored"] = bureau_clean["score"].isnull().astype(int)
bureau_clean["has_no_revolving_limit"] = (bureau_clean["revolving_credit_limit"] <= 0).astype(int)

# Deduplikasi data transaksi belanja pada tingkat nomor pesanan unik
ecommerce_orders_clean = (
    ecommerce.groupby("order_id", as_index=False)
    .agg({
        "customer_id": "first",
        "transaction_datetime": "first",
        "order_total": "first",
        "payment_method": "first",
        "order_status": "first",
        "order_item_id": "count",
    })
    .rename(columns={"order_item_id": "item_count"})
)
ecommerce_orders_clean["is_completed"] = (ecommerce_orders_clean["order_status"] == "completed").astype(int)

cleaning_records = [
    {
        "Tabel": "train_applications",
        "Tindakan": "Konversi kolom tanggal ke format datetime dan validasi urutan waktu",
        "Baris Awal": len(train),
        "Baris Bersih": len(train),
        "Pemeriksaan Nilai Kosong": int(train.isnull().sum().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "test_applications",
        "Tindakan": "Konversi kolom tanggal ke format datetime dan validasi urutan waktu",
        "Baris Awal": len(test),
        "Baris Bersih": len(test),
        "Pemeriksaan Nilai Kosong": int(test.isnull().sum().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "customer_profiles",
        "Tindakan": "Validasi kelengkapan profil dan keunikan nomor nasabah",
        "Baris Awal": len(profiles),
        "Baris Bersih": len(profiles),
        "Pemeriksaan Nilai Kosong": int(profiles.isnull().sum().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "previous_applications",
        "Tindakan": "Pembuatan flag penolakan dan pengisian nol nominal pinjaman",
        "Baris Awal": len(prev_clean),
        "Baris Bersih": len(prev_clean),
        "Pemeriksaan Nilai Kosong": int(prev_clean["approved_amount_clean"].isnull().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "repayment_history",
        "Tindakan": "Konversi tanggal dan pembuatan indikator tagihan belum lunas",
        "Baris Awal": len(rep_clean),
        "Baris Bersih": len(rep_clean),
        "Pemeriksaan Nilai Kosong": int(rep_clean["amount_paid"].isnull().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "account_behavior",
        "Tindakan": "Konversi tanggal bulan dan konfirmasi kepatuhan batas saldo non negatif",
        "Baris Awal": len(account),
        "Baris Bersih": len(account),
        "Pemeriksaan Nilai Kosong": int(account.isnull().sum().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "bureau_observations",
        "Tindakan": "Pembuatan flag penanda nasabah tanpa skor dan batas kredit nol",
        "Baris Awal": len(bureau_clean),
        "Baris Bersih": len(bureau_clean),
        "Pemeriksaan Nilai Kosong": int(bureau_clean["customer_id"].isnull().sum()),
        "Status": "Valid",
    },
    {
        "Tabel": "ecommerce_transactions",
        "Tindakan": "Deduplikasi tingkat pesanan per order id dan hitung jumlah barang belanjaan",
        "Baris Awal": len(ecommerce),
        "Baris Bersih": len(ecommerce_orders_clean),
        "Pemeriksaan Nilai Kosong": int(ecommerce_orders_clean.isnull().sum().sum()),
        "Status": "Valid",
    },
]

cleaning_results_df = pd.DataFrame(cleaning_records)

print("Hasil eksekusi pembersihan data:")
print(cleaning_results_df.to_string(index=False))

### Ringkasan Tindakan Pembersihan Data

| Nama Tabel | Temuan Data Mentah | Dasar Regulasi atau Literatur | Tindakan Pembersihan | Status Validasi |
|---|---|---|---|---|
| train_applications | Format string pada tanggal | Standar perbankan memerlukan tipe datetime | Konversi kolom waktu ke datetime | Berhasil valid tanpa selisih waktu negatif |
| test_applications | Format string pada tanggal | Konsistensi format data latih dan data uji | Konversi kolom waktu ke datetime | Berhasil valid dan konsisten dengan data latih |
| customer_profiles | Profil nasabah perlu dipastikan lengkap | Identitas demografis wajib tersedia untuk skoring | Verifikasi keunikan customer id dan ketiadaan data kosong | Berhasil seluruh nasabah memiliki profil lengkap |
| previous_applications | Kolom persetujuan kosong pada pinjaman ditolak | Alur persetujuan kredit tidak menerbitkan kontrak pinjaman jika ditolak | Buat flag indikator pengajuan tidak disetujui | Berhasil seluruh nilai kosong selaras dengan keputusan |
| repayment_history | Tanggal pembayaran kosong pada cicilan tertunggak | Tagihan belum terbayar belum memiliki bukti kas transaksi | Buat indikator tagihan belum lunas | Berhasil tidak ada nominal pembayaran negatif |
| account_behavior | Pengecekan potensi saldo negatif dari fasilitas cerukan | Literatur kredit mencatat cerukan sebagai utang jangka pendek | Uji nilai minimum seluruh saldo rekening | Berhasil seluruh saldo bernilai nol atau positif |
| bureau_observations | Skor kredit kosong dan utilisasi kosong | Regulasi SLIK mencatat peminjam baru sebagai kelompok tanpa riwayat | Buat flag indikator nasabah tanpa skor dan limit nol | Berhasil nilai kosong terpetakan sebagai sinyal risiko |
| ecommerce_transactions | Duplikasi nilai pesanan pada tiap baris barang | Transaksi mencatat rincian per barang belanjaan | Deduplikasi tingkat pesanan per order id | Berhasil agregasi belanja akurat tanpa penggelembungan nilai |

Insight:
Seluruh dataset kini berada dalam kondisi bersih terstruktur dan siap digunakan untuk proses rekayasa fitur.

In [ ]:
print("Seluruh proses pembersihan data selesai.")
print("Dataset bersih siap digunakan untuk tahap rekayasa fitur.")